<a href="https://colab.research.google.com/github/harini200614/Data-Visualization-lab/blob/main/DVT_4__231401033.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Import Libraries and Load the Real Kaggle Dataset

In [ ]:
!pip -q install kagglehub

import kagglehub
import pandas as pd
import numpy as np
import glob
import os

# Download the real Kaggle dataset
dataset_path = kagglehub.dataset_download(
    "adrianjuliusaluoch/daily-google-search-trends-us"
)

# Find the CSV file automatically
csv_files = glob.glob(
    os.path.join(dataset_path, "**", "*.csv"),
    recursive=True
)

if not csv_files:
    raise FileNotFoundError(
        "No CSV file was found in the downloaded Kaggle dataset."
    )

csv_path = csv_files[0]

# Load the dataset
df = pd.read_csv(csv_path)

print("Dataset loaded successfully!")
print("CSV file:", csv_path)
print("Shape:", df.shape)

print("\nFirst 5 records:")
display(df.head())

## 2. Prepare the Google Trends Columns

In [1]:
# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print("Available columns:")
print(df.columns.tolist())

def find_column(candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    return None

# Detect actual Kaggle columns
query_col = find_column([
    "query", "trends", "trend", "search_query",
    "keyword", "term", "search_term"
])

date_col = find_column([
    "date", "collection_date", "start_time",
    "end_time", "date_recorded", "datetime",
    "timestamp", "time"
])

location_col = find_column([
    "location", "country", "region", "geo"
])

volume_col = find_column([
    "search_volume_lower", "search_volume",
    "volume"
])

if query_col is None:
    raise ValueError(
        "Search-query column was not found. Available columns: "
        + str(df.columns.tolist())
    )

if date_col is None:
    raise ValueError(
        "Date column was not found. Available columns: "
        + str(df.columns.tolist())
    )

print("\nDetected columns:")
print("Query   :", query_col)
print("Date    :", date_col)
print("Location:", location_col)
print("Volume  :", volume_col)

# Create clean query field
df["query"] = df[query_col].astype(str).str.strip()

# Convert date/time field
df["date"] = pd.to_datetime(df[date_col], errors="coerce")

# Extract year
df["year"] = df["date"].dt.year

# Convert search volume values such as 50K+, 2M+ into numbers
def volume_to_number(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().upper()
    text = text.replace(",", "").replace("+", "")

    try:
        if text.endswith("B"):
            return float(text[:-1]) * 1_000_000_000
        elif text.endswith("M"):
            return float(text[:-1]) * 1_000_000
        elif text.endswith("K"):
            return float(text[:-1]) * 1_000
        else:
            return float(text)
    except ValueError:
        return np.nan

if volume_col is not None:
    df["search_volume"] = df[volume_col].apply(volume_to_number)
else:
    # If the source has no search-volume field, create a frequency-based
    # numeric measure from repeated trend records.
    df["search_volume"] = (
        df.groupby("query")["query"]
        .transform("count")
        .astype(float)
    )
    print("\nNo search-volume column was found.")
    print("A frequency-based numeric measure was created for statistical operations.")

# Location
if location_col is not None:
    df["location"] = df[location_col].astype(str).str.strip()
else:
    # This is the US Google Trends dataset.
    df["location"] = "United States"

# Remove rows without usable date/year
df = df.dropna(subset=["year"]).copy()
df["year"] = df["year"].astype(int)

print("\nPrepared dataset:")
display(df[["query", "date", "year", "location", "search_volume"]].head())

NameError: name 'df' is not defined

## 3. Display Shape

In [ ]:
print("Number of rows and columns:")
print(df.shape)

print("\nRows:", df.shape[0])
print("Columns:", df.shape[1])

## 4. Display Data Types

In [ ]:
print("Data types:")
print(df.dtypes)

## 5. Filter Recent Google Trends (Year > 2020)

In [ ]:
recent_trends = df[df["year"] > 2020].copy()

print("Google Trends recorded after 2020:")
print("Number of records:", len(recent_trends))

display(
    recent_trends[
        ["query", "date", "year", "location", "search_volume"]
    ].head(20)
)

## 6. Filter United States Trends

In [ ]:
# The selected Kaggle dataset is the US Google Trends dataset.
# This filter is kept to mirror the country-filtering operation
# from the original Experiment 4.

us_titles = df[
    df["location"].str.lower().isin(
        ["united states", "us", "usa", "united states of america"]
    )
].copy()

print("Google Trends records from the United States:")
print("Number of records:", len(us_titles))

display(
    us_titles[
        ["query", "date", "year", "location", "search_volume"]
    ].head(20)
)

## 7. Sort Google Trends by Search Volume

In [ ]:
sorted_df = df.sort_values(
    by="search_volume",
    ascending=False,
    na_position="last"
)

print("Google Trends sorted by search volume (highest first):")

display(
    sorted_df[
        ["query", "date", "year", "location", "search_volume"]
    ].head(20)
)

## 8. Mean, Median, Mode and Standard Deviation

In [ ]:
volume = df["search_volume"].dropna()

print("Search Volume Statistics")
print("------------------------")

print("Mean =",
      volume.mean())

print("Median =",
      volume.median())

mode_values = volume.mode()

print("Mode =",
      mode_values.iloc[0] if not mode_values.empty else "No unique mode")

print("Standard Deviation =",
      volume.std())

print("\nNumber of valid search-volume values:",
      len(volume))

## 9. Descriptive Statistics

In [ ]:
print("Descriptive statistics for Search Volume:")
display(
    df["search_volume"].describe().to_frame().T
)

## 10. Group By Year – Average Search Volume

In [ ]:
year_avg = (
    df.groupby("year")["search_volume"]
    .mean()
    .sort_index()
)

print("Average Search Volume by Year:")
display(year_avg.to_frame(name="Average_Search_Volume"))

## 11. Group By Year – Number of Trends

In [ ]:
year_count = (
    df.groupby("year")["query"]
    .count()
    .sort_index()
)

print("Number of Google Trends records by Year:")
display(year_count.to_frame(name="Number_of_Trends"))

## 12. Final Summary

In [ ]:
print("FINAL SUMMARY")
print("=============")
print("Dataset shape:", df.shape)
print("Year range:", df["year"].min(), "to", df["year"].max())
print("Total trend records:", len(df))
print("Unique search queries:", df["query"].nunique())
print("Mean search volume:", round(df["search_volume"].mean(), 2))
print("Median search volume:", round(df["search_volume"].median(), 2))

print("\nTop 10 Google Trends by Search Volume:")
display(
    df.sort_values("search_volume", ascending=False)[
        ["query", "date", "year", "location", "search_volume"]
    ].head(10)
)

# Result

The real Kaggle Google Trends dataset was successfully loaded and analyzed using Pandas.

The experiment demonstrates:

- Data loading
- DataFrame shape and data types
- Conditional filtering
- Location filtering
- Sorting
- Mean, median, mode and standard deviation
- Descriptive statistics
- Group-by aggregation
- Final data summary

The original Experiment 4 notebook used the same type of operations on a Netflix dataset. This version applies those operations to the real Google Trends dataset and removes the original machine-specific Windows file path.